# Prosperity 4 — Market Making Strategy Backtest

Compares three strategies on any price CSV in the Prosperity format:

| Strategy | Based on |
|---|---|
| **Naive** | Quote 1 tick inside bot spread, symmetric |
| **AS-Adaptive** | Avellaneda-Stoikov (2008) + Cartea et al. (2017) adaptive σ², κ, OFI |
| **Abernethy-Kale** | Spread-ladder + weighted majority online learning (NeurIPS 2013) |

> **Data used:** `prices_round_0_day_1.csv` and `prices_round_0_day_2.csv` from the Prosperity project data capsule.  
> Change `CONFIG` below to plug in any round's data.


## 1 · Configuration — change these to adapt to any market

In [ ]:
import os

CONFIG = {
    # ── Data ──────────────────────────────────────────────────────────────────
    "price_files": [
        "prices_round_0_day_-1.csv",
        "prices_round_0_day_-2.csv",
    ],
    "data_dir": ".",          # folder containing the CSVs (. = same dir as notebook)

    # ── Products to analyse (must match 'product' column in CSVs) ─────────────
    "products": ["EMERALDS", "TOMATOES"],

    # ── Per-product position limits ───────────────────────────────────────────
    "position_limits": {
        "EMERALDS": 80,
        "TOMATOES": 80,
    },

    # ── Known stable fair values (set to None → estimate live via EMA) ────────
    "fixed_fair": {
        "EMERALDS": 10_000,
        "TOMATOES": None,
    },

    # ── A-S risk-aversion γ per product ───────────────────────────────────────
    "gamma": {
        "EMERALDS": 0.05,
        "TOMATOES": 0.04,
    },

    # ── Misc ──────────────────────────────────────────────────────────────────
    "ema_alpha":    0.25,   # EMA smoothing for live fair-value estimate
    "vol_window":   50,     # rolling window for σ² estimation
    "ofi_weight":   0.30,   # OFI → reservation price tilt
    "quote_qty":    20,     # max units per passive quote
    "ak_spreads":   [1, 2, 3, 4, 5, 7],   # Abernethy-Kale half-spread ladder
    "ak_eta":       0.02,   # weighted-majority learning rate
}

print("Config loaded ✓")
print(f"  Products     : {CONFIG['products']}")
print(f"  Price files  : {CONFIG['price_files']}")


## 2 · Imports

In [ ]:
import csv, math, statistics, os
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Callable, Optional

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})
print("Imports OK ✓")


## 3 · Data structures

In [ ]:
@dataclass
class Snap:
    """One row from the prices CSV — one product at one timestamp."""
    ts:      int
    product: str
    bid1: float; bvol1: int
    bid2: float; bvol2: int
    ask1: float; avol1: int
    ask2: float; avol2: int
    mid:  float

    @property
    def bot_half_spread(self) -> float:
        return (self.ask1 - self.bid1) / 2.0


@dataclass
class Fill:
    ts:      int
    price:   float
    qty:     int      # positive = buy, negative = sell
    passive: bool


@dataclass
class Result:
    name:    str
    product: str
    pnl:     List[float] = field(default_factory=list)
    pos:     List[int]   = field(default_factory=list)
    fills:   List[Fill]  = field(default_factory=list)

    def total_pnl(self):  return self.pnl[-1] if self.pnl else 0.0
    def max_pos(self):    return max((abs(p) for p in self.pos), default=0)
    def n_trades(self):   return len(self.fills)
    def sharpe(self):
        diffs = [self.pnl[i] - self.pnl[i-1] for i in range(1, len(self.pnl))]
        if len(diffs) < 2: return 0.0
        mu = sum(diffs) / len(diffs)
        sd = statistics.stdev(diffs)
        return (mu / sd) * math.sqrt(len(diffs)) if sd > 0 else 0.0

print("Data structures defined ✓")


## 4 · Data loader

In [ ]:
def load_snaps(product: str, cfg: dict) -> List[Snap]:
    """Load all snapshots for *product* from the configured price CSVs."""
    snaps = []
    for fname in cfg["price_files"]:
        path = os.path.join(cfg["data_dir"], fname)
        if not os.path.exists(path):
            print(f"  ⚠  File not found: {path}")
            continue
        with open(path) as f:
            for row in csv.DictReader(f, delimiter=";"):
                if row["product"] != product: continue
                fv = lambda k: float(row[k]) if row.get(k) else 0.0
                iv = lambda k: int(float(row[k])) if row.get(k) else 0
                snaps.append(Snap(
                    ts=int(row["timestamp"]), product=product,
                    bid1=fv("bid_price_1"), bvol1=iv("bid_volume_1"),
                    bid2=fv("bid_price_2"), bvol2=iv("bid_volume_2"),
                    ask1=fv("ask_price_1"), avol1=iv("ask_volume_1"),
                    ask2=fv("ask_price_2"), avol2=iv("ask_volume_2"),
                    mid=fv("mid_price"),
                ))
    print(f"  {product}: loaded {len(snaps):,} snapshots")
    return snaps


# Quick preview
for prod in CONFIG["products"]:
    snaps = load_snaps(prod, CONFIG)
    if snaps:
        s = snaps[0]
        print(f"    first snap → ts={s.ts}  mid={s.mid}  "
              f"bot book={s.bid1}/{s.ask1}  spread={s.ask1-s.bid1:.1f}")


## 5 · Fill simulation engine

In [ ]:
def simulate_fills(snap: Snap, our_bid: float, our_ask: float,
                   position: int, quote_qty: int,
                   pos_limit: int, rng_seed: float = 0.0) -> List[Fill]:
    """
    Simulate order fills for one timestep.

    Aggressive fills: our quote crosses the bot's best price → immediate fill.
    Passive fills   : our quote is inside the spread → fill with probability
                      proportional to how close we are to the aggressive side.
    """
    fills, pos = [], position

    # Aggressive buy
    if our_bid >= snap.ask1 and pos < pos_limit:
        qty = min(quote_qty, snap.avol1, pos_limit - pos)
        if qty > 0:
            fills.append(Fill(snap.ts, snap.ask1, qty, passive=False))
            pos += qty

    # Aggressive sell
    if our_ask <= snap.bid1 and pos > -pos_limit:
        qty = min(quote_qty, snap.bvol1, pos + pos_limit)
        if qty > 0:
            fills.append(Fill(snap.ts, snap.bid1, -qty, passive=False))
            pos -= qty

    spread = snap.ask1 - snap.bid1

    # Passive bid
    if snap.bid1 < our_bid < snap.ask1 and pos < pos_limit:
        prob   = (our_bid - snap.bid1) / spread if spread > 0 else 0
        pseudo = abs(math.sin(rng_seed * 12.9898 + our_bid * 78.233) * 43758.5453) % 1
        if pseudo < prob:
            qty = min(quote_qty, pos_limit - pos)
            if qty > 0:
                fills.append(Fill(snap.ts, our_bid, qty, passive=True))

    # Passive ask
    if snap.bid1 < our_ask < snap.ask1 and pos > -pos_limit:
        prob   = (snap.ask1 - our_ask) / spread if spread > 0 else 0
        pseudo = abs(math.sin(rng_seed * 78.233 + our_ask * 12.9898) * 43758.5453) % 1
        if pseudo < prob:
            qty = min(quote_qty, pos + pos_limit)
            if qty > 0:
                fills.append(Fill(snap.ts, our_ask, -qty, passive=True))

    return fills


def run_strategy(snaps: List[Snap], strategy_fn: Callable,
                 name: str, cfg: dict) -> Result:
    """Drive a strategy through all snapshots and collect PnL / positions."""
    product   = snaps[0].product
    pos_limit = cfg["position_limits"].get(product, 20)
    quote_qty = cfg["quote_qty"]
    result    = Result(name=name, product=product)
    state     = {}
    cash, pos = 0.0, 0

    for i, snap in enumerate(snaps):
        our_bid, our_ask = strategy_fn(snap, pos, state, cfg)
        if our_bid >= our_ask:
            our_ask = our_bid + 1

        for f in simulate_fills(snap, our_bid, our_ask, pos,
                                 quote_qty, pos_limit, rng_seed=float(i)):
            cash -= f.price * f.qty
            pos  += f.qty
            result.fills.append(f)

        result.pnl.append(cash + pos * snap.mid)
        result.pos.append(pos)

    return result

print("Fill engine defined ✓")


## 6 · Strategies

Each strategy is a pure function `(snap, position, state_dict, config) → (bid, ask)`.

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def get_fair(snap: Snap, state: dict, cfg: dict) -> float:
    """EMA-based fair value, or fixed if configured."""
    fixed = cfg["fixed_fair"].get(snap.product)
    if fixed is not None:
        return float(fixed)
    alpha = cfg["ema_alpha"]
    prev  = state.get("ema", snap.mid)
    fair  = alpha * snap.mid + (1 - alpha) * prev
    state["ema"] = fair
    return fair


def rolling_sigma2(snap: Snap, state: dict, cfg: dict) -> float:
    """Rolling variance of mid-price changes."""
    hist = state.setdefault("mids", [])
    hist.append(snap.mid)
    win = cfg["vol_window"] + 1
    if len(hist) > win: del hist[:-win]
    if len(hist) >= 2:
        ch = [hist[i+1] - hist[i] for i in range(len(hist)-1)]
        mu = sum(ch) / len(ch)
        return max(sum((c - mu)**2 for c in ch) / len(ch), 0.25)
    # seed from known calibration while window fills
    seeds = {"EMERALDS": 1.03, "TOMATOES": 1.89}
    return seeds.get(snap.product, 1.5)


def ofi(snap: Snap) -> float:
    """Order-flow imbalance ∈ [-1, +1]."""
    b = snap.bvol1 + snap.bvol2
    a = snap.avol1 + snap.avol2
    t = b + a
    return (b - a) / t if t > 0 else 0.0


# ── 1. Naive ───────────────────────────────────────────────────────────────────
def strategy_naive(snap, pos, state, cfg):
    """Quote 1 tick inside the bot spread. No inventory awareness."""
    bid = math.floor(snap.bid1 + 1)
    ask = math.ceil (snap.ask1 - 1)
    if bid >= ask:
        bid = math.floor(snap.mid) - 1
        ask = math.ceil (snap.mid) + 1
    return bid, ask


# ── 2. AS-Adaptive ─────────────────────────────────────────────────────────────
def strategy_as_adaptive(snap, pos, state, cfg):
    """
    Avellaneda-Stoikov (2008) reservation price + optimal spread,
    with adaptive σ² (rolling window), adaptive κ (live from bot spread),
    and OFI signal tilt (Cartea et al. 2017).

        r = fair - q·γ·σ²  +  α_OFI · OFI
        δ = clip( (γσ² + (2/γ)·ln(1+γ/κ)) / 2 ,  [1, 5] )
    """
    fair  = get_fair(snap, state, cfg)
    s2    = rolling_sigma2(snap, state, cfg)
    g     = cfg["gamma"].get(snap.product, 0.05)

    # Live κ from current bot half-spread
    hbs   = snap.bot_half_spread
    kappa = max(0.5, min(math.log(2) / hbs if hbs > 0 else 1.5, 3.0))

    r = fair - pos * g * s2 + cfg["ofi_weight"] * ofi(snap)
    h = max(1, min((g * s2 + (2/g) * math.log(1 + g/kappa)) / 2, 5))

    return math.floor(r - h), math.ceil(r + h)


# ── 3. Abernethy-Kale ──────────────────────────────────────────────────────────
def strategy_abernethy_kale(snap, pos, state, cfg):
    """
    Spread-ladder with exponential-weights online learning (NeurIPS 2013).

    A set of candidate half-spreads is maintained. After each step the weight
    of each spread is updated proportionally to the profit it would have earned,
    penalised for inventory build-up.  The chosen δ is the weighted average.
    The reservation price uses A-S inventory adjustment so the centre is
    inventory-aware.
    """
    spreads = cfg["ak_spreads"]
    eta     = cfg["ak_eta"]
    n       = len(spreads)

    w = state.setdefault("weights", [1.0/n]*n)
    fair = get_fair(snap, state, cfg)
    s2   = rolling_sigma2(snap, state, cfg)
    g    = cfg["gamma"].get(snap.product, 0.05)

    pos_limit   = cfg["position_limits"].get(snap.product, 20)
    inv_penalty = abs(pos) / pos_limit

    # Multiplicative weights update: reward wider spreads less when long/short
    gains = [d * (1.0 - inv_penalty) for d in spreads]
    new_w = [w[i] * math.exp(eta * gains[i]) for i in range(n)]
    z     = sum(new_w)
    state["weights"] = [x/z for x in new_w]

    chosen = sum(state["weights"][i] * spreads[i] for i in range(n))
    h      = max(1, min(round(chosen), 5))

    # A-S reservation price as the centre
    r = fair - pos * g * s2

    return math.floor(r - h), math.ceil(r + h)


STRATEGIES = [
    ("Naive",           strategy_naive),
    ("AS-Adaptive",     strategy_as_adaptive),
    ("Abernethy-Kale",  strategy_abernethy_kale),
]

print(f"{len(STRATEGIES)} strategies registered:")
for name, _ in STRATEGIES:
    print(f"  • {name}")


## 7 · Run backtest

In [ ]:
all_results: Dict[str, Dict[str, Result]] = {}

for product in CONFIG["products"]:
    snaps = load_snaps(product, CONFIG)
    if not snaps:
        print(f"  No data for {product}, skipping")
        continue

    print(f"\n{'='*62}")
    print(f"  {product}  ({len(snaps):,} snapshots)")
    print(f"{'='*62}")
    print(f"  {'Strategy':<22} {'PnL':>12} {'Trades':>8} {'MaxPos':>8} {'Sharpe':>8}")
    print(f"  {'-'*60}")

    all_results[product] = {}
    for name, fn in STRATEGIES:
        res = run_strategy(snaps, fn, name, CONFIG)
        all_results[product][name] = res
        print(f"  {name:<22} {res.total_pnl():>12,.1f} "
              f"{res.n_trades():>8,} {res.max_pos():>8} {res.sharpe():>8.3f}")

print("\n✓ Done")


## 8 · Fill type breakdown

In [ ]:
print(f"{'Product':<12} {'Strategy':<22} {'Aggressive':>12} {'Passive':>10} {'Total':>8}")
print("-"*68)
for product, results in all_results.items():
    for name, res in results.items():
        agg = sum(1 for f in res.fills if not f.passive)
        pas = sum(1 for f in res.fills if f.passive)
        print(f"{product:<12} {name:<22} {agg:>12,} {pas:>10,} {agg+pas:>8,}")


## 9 · Cumulative PnL

In [ ]:
COLORS = {
    "Naive":          "#6c757d",
    "AS-Adaptive":    "#198754",
    "Abernethy-Kale": "#dc3545",
}

n_products = len(all_results)
fig, axes = plt.subplots(1, n_products, figsize=(7 * n_products, 4), sharey=False)
if n_products == 1: axes = [axes]

for ax, (product, results) in zip(axes, all_results.items()):
    for name, res in results.items():
        ax.plot(res.pnl, label=name, color=COLORS.get(name, "blue"),
                linewidth=1.6, alpha=0.9)
    ax.set_title(f"{product} — Cumulative PnL", fontweight="bold")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("XIRECs")
    ax.legend()

plt.tight_layout()
plt.show()


## 10 · Inventory over time

In [ ]:
fig, axes = plt.subplots(1, n_products, figsize=(7 * n_products, 4), sharey=False)
if n_products == 1: axes = [axes]

for ax, (product, results) in zip(axes, all_results.items()):
    pos_limit = CONFIG["position_limits"].get(product, 20)
    for name, res in results.items():
        ax.plot(res.pos, label=name, color=COLORS.get(name, "blue"),
                linewidth=1.2, alpha=0.8)
    ax.axhline( pos_limit, color="black", ls="--", lw=0.8, label="±Limit")
    ax.axhline(-pos_limit, color="black", ls="--", lw=0.8)
    ax.set_title(f"{product} — Inventory", fontweight="bold")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Units held")
    ax.legend()

plt.tight_layout()
plt.show()


## 11 · Summary — Final PnL & Sharpe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric in zip(axes, ["PnL", "Sharpe"]):
    x     = list(range(len(STRATEGIES)))
    width = 0.25
    for i, (product, results) in enumerate(all_results.items()):
        names  = [n for n,_ in STRATEGIES if n in results]
        vals   = ([results[n].total_pnl() for n in names]
                  if metric == "PnL" else
                  [results[n].sharpe()    for n in names])
        colors = [COLORS.get(n, "blue") for n in names]
        offset = (i - (n_products-1)/2) * width
        bars   = ax.bar([xi + offset for xi in range(len(names))], vals,
                        width=width, label=product, color=colors, alpha=0.85,
                        edgecolor="white")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() * 1.01,
                    f"{val:,.0f}" if metric == "PnL" else f"{val:.2f}",
                    ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(range(len(STRATEGIES)))
    ax.set_xticklabels([n for n,_ in STRATEGIES], rotation=15, ha="right")
    ax.set_title(f"Final {metric} by Strategy", fontweight="bold")
    ax.set_ylabel(metric)
    ax.axhline(0, color="black", lw=0.8)
    if n_products > 1: ax.legend(title="Product")

plt.tight_layout()
plt.show()


## 12 · Interpretation

### What each metric tells you

| Metric | What it measures |
|---|---|
| **PnL** | Raw profit — the main competition score |
| **Sharpe** | Risk-adjusted return — high means consistent, low means volatile swings |
| **Max position** | How close to the ±limit you got — hitting the limit cancels all orders |
| **Trade count** | More trades = more spread capture, but also more inventory risk |

### Using this for new rounds

1. **Update `CONFIG["price_files"]`** to point at the new round's CSV files  
2. **Update `CONFIG["products"]`** and `position_limits` to match the round's products  
3. **Set `CONFIG["fixed_fair"]`** to `None` for any product without a stable fair value (EMA will be used)  
4. Re-run cells 7–11  

The strategy functions themselves require **no changes** — they read all parameters from `CONFIG`.

### Strategy selection guide

- **Stable, mean-reverting asset** (like EMERALDS): both A-S Adaptive and Abernethy-Kale work well; A-K tends to find the right spread faster  
- **Trending/drifting asset** (like TOMATOES): A-S Adaptive dominates because the reservation-price adjustment from rolling σ² and OFI corrects the centre before you accumulate inventory on the wrong side  
- **Unknown asset**: start with A-S Adaptive as the safer default; inspect the inventory chart — if it trends toward the limit, tighten γ or widen the OFI weight
